In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torch.optim import Adam
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

X_train = torch.from_numpy(X_train).float()
X_test = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).float()
y_test = torch.from_numpy(y_test).float().unsqueeze(1)


print(f"X_train tensor shape: {X_train.shape}")

print(f"X_test tensor shape: {X_test.shape}")
print(f"y_train tensor shape: {y_train.shape}")
print(f"y_test tensor shape: {y_test.shape}")




In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)




In [ ]:
# 3. Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)




In [ ]:
# 4. Print shape of one batch
for batch_idx, (data, target) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1} - Data shape: {data.shape}, Target shape: {target.shape}")
    break



In [ ]:
# 5. Display sample images
def display_images(images, num_images=5):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))



In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim):

        super(NN4Layer, self).__init__()

        # input_dim = num of features, hidden_dim = num of neurons
        self.layer1 = nn.Linear(input_dim, hidden_dim)# output of 1st layer = input of 2nd layer
        # hidden_dim = num of neurons, Output for binary classification is 1
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):
          z1 = self.layer1(x)#نمرر الانبوت على اللاير الاول
          a1 = self.relu(z1)#لازم نفس الترتيب ---- layer >> activation function
          z2 = self.layer2(a1)
          a2 = self.relu(z2)
          z3 = self.layer3(a2)
          a3 = self.relu(z3)
          z4 = self.layer4(a3)
          a4 = self.sigmoid(z4)




          return a4





In [ ]:
# Task 2: Write your training loop here:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(model, train_loader, optimizer, criterion):

    model.train()
    model.to(device)


    train_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):

        optimizer.zero_grad()

        output = model(data)


        loss = criterion(output, target)
        loss.backward()
        optimizer.step()


        train_loss += loss.item()


    return train_loss / len(train_loader)

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for X_batch, y_batch in test_loader:
            # Move batch to the selected device
            X_batch = X_batch.to(device)
            y_batch = y_batch.view(-1, 1).to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()# ask him

            # Binary predictions
            predicted = (outputs > 0.5).float()

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = X_train.shape[1]
hidden_dim = 128
model = NN4Layer(input_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)



In [ ]:
import torch
import torch.nn as nn

# Create a linear layer: 3 input features -> 1 output
linear = nn.Linear(in_features=3, out_features=1)#***

# Example input: batch of 4 samples, each with 3 features
x = torch.randn(4, 3)

# Forward through the layer
y = linear(x)

print("Input shape:", x.shape)   # torch.Size([4, 3])
print("Output shape:", y.shape) # torch.Size([4, 1])
print(y)

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()
    model.to(device)

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1, 1).to(device)

        # Forward pass
        outputs = model(X_batch)#asl him
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients---impo!! forgite previous
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
num_epochs = 20  #*** proportional to the time
learning_rate = 0.001

# Model, Criterion, Optimizer
input_dim = X_train.shape[1]# num of feature or column
hidden_dim = 10  #neorons
model =NN4Layer(input_dim, hidden_dim).to(device)
criterion = nn.BCELoss()  #loss function  for two class
optimizer = Adam(model.parameters(), lr=learning_rate)


# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if (epoch + 1) % 5 == 0: # every 5 epoch print the result
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []
for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, criterion, test_loader, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')

In [ ]:
# Task 2 (Bonus): Write your code here: